In [ ]:
import os, sys, json, time, hashlib, math
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
CONFIG_PATH = os.path.join(BASE_DIR, "config.json")
config = json.load(open(CONFIG_PATH))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR, FEATURES_DIR, RESULTS_DIR = config["paths"]["data_dir"], config["paths"]["features_dir"], config["paths"]["results_dir"]
EMOTIONS, C = config["emotions"], len(config["emotions"])
SPLITS = ["train", "dev", "test"]
ENABLED = os.environ.get("MELD_TEXT_FT", "1").strip().lower() in ("1", "true", "yes")
K = int(os.environ.get("MELD_P3B_FOLDS", 5))
EPOCHS = int(os.environ.get("MELD_P3B_EPOCHS", 3))             
_FT_ENV = os.environ.get("MELD_P3B_MODEL", "")
LR = float(os.environ.get("MELD_P3B_LR", 1e-5 if "large" in _FT_ENV else 2e-5))
BATCH = int(os.environ.get("MELD_P3B_BATCH", 16 if "large" in _FT_ENV else 32))
MAX_LEN = 256
MAX_TGT = 96                                                      
CTX_N = int(os.environ.get("MELD_TEXT_CTX", 3))                 
assert CTX_N >= 1, "Version A needs MELD_TEXT_CTX >= 1 (use the v4 Phase_03b for no context)"
WEIGHT_POWER = float(os.environ.get("MELD_P3B_WEIGHT_POWER", 0.5))   
SEED = int(config["seed"])
TEXT_MODEL = config["text_model"]                                  
FT_MODEL = os.environ.get("MELD_P3B_MODEL", TEXT_MODEL)             
IS_LARGE = "large" in FT_MODEL
FT_FILE = "text_ft_features_{split}.npy"
META_PATH, REPORT_PATH, VERIF_PATH = os.path.join(FEATURES_DIR, "text_ft_meta.json"), os.path.join(RESULTS_DIR, "text_ft_report.json"), os.path.join(RESULTS_DIR, "text_ft_verification.json")
CACHE_DIR = os.path.join(FEATURES_DIR, "_p3b_folds"); os.makedirs(CACHE_DIR, exist_ok=True)
SETTINGS = {"model": FT_MODEL, "ft_model": FT_MODEL, "frozen_model": TEXT_MODEL, "folds": K, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "max_len": MAX_LEN, "weight_power": WEIGHT_POWER, "seed": SEED,
            "pooling": "mean_over_target_tokens", "context_utterances": CTX_N, "context_direction": "past_only_same_dialogue", "input_format": "Speaker: text; <s> ctx </s></s> target </s>", "max_tgt": MAX_TGT, "dropout": 0.1, "warmup": 0.1, "weight_decay": 0.01}
SETTINGS_ID = hashlib.sha1(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:10]

def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

if not ENABLED:
    for key in ("text_feature_file", "text_fusion_feature_dim", "text_feature_set"):
        config.pop(key, None)
    write_json(CONFIG_PATH, config)
    write_json(META_PATH, {"enabled": False})
    write_json(REPORT_PATH, {"enabled": False, "note": "MELD_TEXT_FT=0: frozen Phase 3 text features are used"})
    write_json(VERIF_PATH, {"n_tests": 0, "n_failed": 0, "results": []})
    print("MELD_TEXT_FT=0: fine-tuned text features are OFF; config.json points to the frozen Phase 3 features. Remaining cells are skipped.")
else:
    print(f"Device: {device} | fine-tuned model: {FT_MODEL} (frozen features: {TEXT_MODEL}) | folds: {K} | epochs: {EPOCHS} | lr: {LR} | batch: {BATCH} | weight power: {WEIGHT_POWER} | settings id: {SETTINGS_ID}")

In [ ]:
def assign_folds(labels, groups, k, seed):
    fold = np.full(len(labels), -1, dtype=np.int64)
    for f, (_, ho) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(np.zeros(len(labels)), labels, groups)):
        fold[ho] = f
    return fold

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64); z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))

def zscore_fit(x):
    mu, sd = x.mean(0), x.std(0)
    return mu, np.where(sd < 1e-8, 1.0, sd)

def class_weights_np(labels, c, power):
    cnt = np.bincount(labels, minlength=c).astype(np.float64)
    w = (len(labels) / (c * np.maximum(cnt, 1))) ** power
    return w / (w[labels]).mean()                     

def macro_f1_np(y, p, c):
    conf = np.bincount(np.asarray(y) * c + np.asarray(p), minlength=c * c).reshape(c, c).astype(np.float64)
    tp, row, col = np.diag(conf), conf.sum(1), conf.sum(0)
    return float(np.where(row + col > 0, 2 * tp / np.maximum(row + col, 1e-300), 0.0).mean())
def context_rows(dialogue_ids, n_ctx):
    """For every row, the row indices of its context: the previous <= n_ctx rows of the SAME dialogue (CSV row order), oldest first."""
    seen, out = {}, []
    for i, d in enumerate(dialogue_ids):
        prev = seen.setdefault(d, [])
        out.append(prev[-n_ctx:] if n_ctx > 0 else [])
        prev.append(i)
    return out

def assemble_ids(ctx_ids, tgt_ids, cls_id, sep_id, max_len, max_tgt):
    """<s> ctx1 </s> ctx2 </s> ... </s></s> target </s>  (no context: <s> target </s>). The context is cut from the LEFT (oldest tokens first).
    Returns (input_ids, target_mask) with target_mask = 1 on the target tokens only."""
    tgt = list(tgt_ids)[:max(1, min(max_tgt, max_len - 2))]
    flat = []
    for k, c in enumerate(ctx_ids):
        flat += list(c) + ([sep_id] if k < len(ctx_ids) - 1 else [])
    n_special = 4 if flat else 2                                  
    room = max(0, max_len - len(tgt) - n_special)
    flat = flat[-room:] if room > 0 else []
    if flat:
        ids = [cls_id] + flat + [sep_id, sep_id] + tgt + [sep_id]; start = 1 + len(flat) + 2
    else:
        ids = [cls_id] + tgt + [sep_id]; start = 1
    mask = [0] * start + [1] * len(tgt) + [0]
    assert len(ids) == len(mask) and len(ids) <= max_len, (len(ids), max_len)
    return ids, mask
print("Helpers defined.")

In [ ]:
if ENABLED:
    dfs = {s: pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv"), keep_default_na=False) for s in SPLITS}
    emo2id = {e: i for i, e in enumerate(EMOTIONS)}
    labels = {s: dfs[s]["Emotion"].str.lower().map(emo2id).values for s in SPLITS}
    for s in SPLITS:
        assert not pd.isnull(labels[s]).any(), f"[{s}] unknown emotion labels"
        labels[s] = labels[s].astype(np.int64)
    texts = {s: dfs[s]["Utterance"].astype(str).tolist() for s in SPLITS}
    groups_train = dfs["train"]["Dialogue_ID"].values
    FOLD = assign_folds(labels["train"], groups_train, K, SEED)
    tok = AutoTokenizer.from_pretrained(FT_MODEL)
    for s in SPLITS:
        assert {"Speaker", "Dialogue_ID"} <= set(dfs[s].columns), f"[{s}] the CSV needs the Speaker and Dialogue_ID columns"
    tagged = {s: [f"{sp}: {t}" for sp, t in zip(dfs[s]["Speaker"].astype(str), texts[s])] for s in SPLITS}
    tok_ids = {s: tok(tagged[s], add_special_tokens=False)["input_ids"] for s in SPLITS}      # tokenised once per utterance
    CTX = {s: context_rows(dfs[s]["Dialogue_ID"].values, CTX_N) for s in SPLITS}
    ids = {s: [assemble_ids([tok_ids[s][j] for j in CTX[s][i]], tok_ids[s][i], tok.cls_token_id, tok.sep_token_id, MAX_LEN, MAX_TGT) for i in range(len(texts[s]))] for s in SPLITS}
    print(f"context: previous {CTX_N} utterances of the same dialogue (past only) | mean tokens per input: " + ", ".join(f"{s} {np.mean([len(x[0]) for x in ids[s]]):.0f}" for s in SPLITS))
    print("fold sizes (utterances):", np.bincount(FOLD).tolist(), "| dialogues per fold:", [int(len(np.unique(groups_train[FOLD == f]))) for f in range(K)])
    print("emotion share per fold (rows: folds, columns: " + ",".join(EMOTIONS) + "):")
    print(np.round(np.array([np.bincount(labels["train"][FOLD == f], minlength=C) / (FOLD == f).sum() for f in range(K)]), 3))

In [ ]:
if ENABLED:
    class TextClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = AutoModel.from_pretrained(FT_MODEL, add_pooling_layer=False) if "roberta" in FT_MODEL else AutoModel.from_pretrained(FT_MODEL)
            if IS_LARGE:
                self.enc.gradient_checkpointing_enable()                      
            self.drop = nn.Dropout(0.1)
            self.head = nn.Linear(self.enc.config.hidden_size, C)
        def forward(self, input_ids, attention_mask, pool_mask):
            h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            m = pool_mask.unsqueeze(-1).to(h.dtype)                                    
            return self.head(self.drop((h * m).sum(1) / m.sum(1).clamp(min=1.0)))      

    def collate(items):                                             
        longest = max(len(x[0]) for x in items)
        inp = torch.full((len(items), longest), tok.pad_token_id, dtype=torch.long); att = torch.zeros((len(items), longest), dtype=torch.long); pm = torch.zeros((len(items), longest), dtype=torch.long)
        for i, (x, tm) in enumerate(items):
            inp[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1; pm[i, :len(x)] = torch.tensor(tm)
        return inp.to(device), att.to(device), pm.to(device)

    @torch.no_grad()
    def predict(model, id_lists, bs=128):
        model.eval(); order = np.argsort([len(x[0]) for x in id_lists]); out = np.zeros((len(id_lists), C), dtype=np.float32)
        for i in range(0, len(order), bs):
            sel = order[i:i + bs]; inp, att, pm = collate([id_lists[j] for j in sel])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                out[sel] = model(inp, att, pm).float().cpu().numpy()
        return out

    def train_fold(f):
        tr = np.flatnonzero(FOLD != f); ho = np.flatnonzero(FOLD == f)
        assert len(set(groups_train[tr]) & set(groups_train[ho])) == 0, "a dialogue is in both the training part and the held-out part"
        torch.manual_seed(SEED + f); np.random.seed(SEED + f)
        model = TextClassifier().to(device)
        w = torch.tensor(class_weights_np(labels["train"][tr], C, WEIGHT_POWER), dtype=torch.float32, device=device)
        lossf = nn.CrossEntropyLoss(weight=w)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        steps = EPOCHS * math.ceil(len(tr) / BATCH)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        use_amp = device.type == "cuda"
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        ytr = torch.tensor(labels["train"], dtype=torch.long)
        for ep in range(EPOCHS):
            model.train(); rng = np.random.RandomState(SEED * 1000 + f * 10 + ep); perm = rng.permutation(tr); tot, n = 0.0, 0
            for i in tqdm(range(0, len(perm), BATCH), desc=f"fold {f} epoch {ep + 1}/{EPOCHS}", leave=False):
                sel = perm[i:i + BATCH]; inp, att, pm = collate([ids["train"][j] for j in sel]); y = ytr[sel].to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    logits = model(inp, att, pm)
                loss = lossf(logits.float(), y)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
                tot += float(loss) * len(sel); n += len(sel)
            print(f"  fold {f} epoch {ep + 1}: train loss {tot / n:.4f}")
        res = {"ho_idx": ho, "ho_logits": predict(model, [ids["train"][j] for j in ho]), "dev_logits": predict(model, ids["dev"]), "test_logits": predict(model, ids["test"])}
        del model, opt; torch.cuda.empty_cache()
        return res
    print("Model / training functions defined.")

In [ ]:
if ENABLED:
    FOLD_RES, t_start = {}, time.time()
    for f in range(K):
        cache = os.path.join(CACHE_DIR, f"fold{f}_{SETTINGS_ID}.npz")
        if os.path.exists(cache):
            z = np.load(cache); FOLD_RES[f] = {k: z[k] for k in ("ho_idx", "ho_logits", "dev_logits", "test_logits")}
            print(f"[fold {f}] found cached result ({cache}) -> skipped"); continue
        t0 = time.time(); FOLD_RES[f] = train_fold(f)
        np.savez(cache + ".tmp.npz", **FOLD_RES[f]); os.replace(cache + ".tmp.npz", cache)
        ho = FOLD_RES[f]["ho_idx"]; p = FOLD_RES[f]["ho_logits"].argmax(1); y = labels["train"][ho]
        print(f"[fold {f}] {(time.time() - t0) / 60:.1f} min | held-out acc {(p == y).mean():.4f} macro-F1 {macro_f1_np(y, p, C):.4f} | dev acc {(FOLD_RES[f]['dev_logits'].argmax(1) == labels['dev']).mean():.4f}")
    print(f"All folds done in {(time.time() - t_start) / 60:.1f} min")

In [ ]:
if ENABLED:
    n_tr = len(labels["train"])
    oof = np.zeros((n_tr, C), dtype=np.float64); seen = np.zeros(n_tr, dtype=int)
    for f in range(K):
        oof[FOLD_RES[f]["ho_idx"]] = log_softmax_np(FOLD_RES[f]["ho_logits"]); seen[FOLD_RES[f]["ho_idx"]] += 1
    assert (seen == 1).all(), "every training row must receive exactly one out-of-fold prediction"
    lp = {"train": oof,
          "dev": np.mean([log_softmax_np(FOLD_RES[f]["dev_logits"]) for f in range(K)], axis=0),
          "test": np.mean([log_softmax_np(FOLD_RES[f]["test_logits"]) for f in range(K)], axis=0)}
    MU, SD = zscore_fit(lp["train"])                   
    frozen = {s: np.load(os.path.join(FEATURES_DIR, f"text_features_{s}.npy")) for s in SPLITS}
    for s in SPLITS:
        assert frozen[s].ndim == 2 and frozen[s].shape == (len(dfs[s]), config["text_feature_dim"]), f"[{s}] Phase 3 features have shape {frozen[s].shape}; run Phase 3 first (the frozen file must be the plain 768-d one)"
    FT = {s: np.concatenate([frozen[s], ((lp[s] - MU) / SD).astype(np.float32)], axis=1).astype(np.float32) for s in SPLITS}
    DIM = int(FT["train"].shape[1])
    rep = {"settings": SETTINGS, "settings_id": SETTINGS_ID, "dim": DIM, "frozen_dim": int(config["text_feature_dim"]), "logit_dims": C,
           "oof_train": {"accuracy": float((oof.argmax(1) == labels["train"]).mean()), "macro_f1": macro_f1_np(labels["train"], oof.argmax(1), C)},
           "dev_kmodel_average": {"accuracy": float((lp["dev"].argmax(1) == labels["dev"]).mean()), "macro_f1": macro_f1_np(labels["dev"], lp["dev"].argmax(1), C)},
           "per_fold_dev_macro_f1": [macro_f1_np(labels["dev"], FOLD_RES[f]["dev_logits"].argmax(1), C) for f in range(K)],
           "train_majority_rate": float(np.bincount(labels["train"], minlength=C).max() / n_tr),
           "note": "test rows get the K-model average like dev; test labels are never read in this notebook"}
    print(json.dumps(rep, indent=2))

In [ ]:
if ENABLED:
    res = []
    def check(name, cond, detail=""):
        res.append({"test": name, "passed": bool(cond), "detail": str(detail)}); print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
    ho_sets = [set(FOLD_RES[f]["ho_idx"].tolist()) for f in range(K)]
    check("W1 held-out parts are disjoint and cover every training row once", sum(len(h) for h in ho_sets) == n_tr and len(set().union(*ho_sets)) == n_tr)
    check("W2 no dialogue appears in two folds", all(len(set(groups_train[FOLD == a]) & set(groups_train[FOLD == b])) == 0 for a in range(K) for b in range(a + 1, K)))
    check("W3 every fold model was trained without its held-out dialogues (train_fold asserts this per fold)", True, "enforced by an assert inside train_fold")
    check("W4 out-of-fold log-probabilities are valid", np.isfinite(oof).all() and np.allclose(np.exp(oof).sum(1), 1.0, atol=1e-6))
    check("W5 OOF accuracy beats the majority-class rate (fine-tuning worked)", rep["oof_train"]["accuracy"] > rep["train_majority_rate"], f"OOF acc {rep['oof_train']['accuracy']:.3f} vs majority {rep['train_majority_rate']:.3f}")
    check("W6 fine-tuned signal is not trivially perfect (would indicate leakage)", rep["oof_train"]["accuracy"] < 0.90 and rep["dev_kmodel_average"]["accuracy"] < 0.90)
    check("W7 frozen part of the output equals the Phase 3 file exactly", all(np.array_equal(FT[s][:, :config["text_feature_dim"]], frozen[s]) for s in SPLITS))
    zt = FT["train"][:, config["text_feature_dim"]:]
    check("W8 z-scored log-probabilities have mean 0 / std 1 on the training rows", np.allclose(zt.mean(0), 0, atol=1e-4) and np.allclose(zt.std(0), 1, atol=1e-3))
    check("W9 all features finite; shapes match the CSV rows", all(np.isfinite(FT[s]).all() and FT[s].shape == (len(dfs[s]), DIM) for s in SPLITS), DIM)
    check("W10 label counts per class match the CSV (train)", np.array_equal(np.bincount(labels["train"], minlength=C), dfs["train"]["Emotion"].str.lower().map(emo2id).value_counts().sort_index().reindex(range(C), fill_value=0).values))
    def _ctx_ok(split):
        d = dfs[split]["Dialogue_ID"].values
        return all(all(j < i and d[j] == d[i] for j in CTX[split][i]) and len(CTX[split][i]) <= CTX_N for i in range(len(d)))
    check("C1 context rows are earlier rows of the same dialogue (past only), at most CTX_N, in every split", all(_ctx_ok(s) for s in SPLITS))
    check("C2 every input has >= 1 target token and the target is the LAST segment", all(sum(m) >= 1 and m[-1] == 0 and m[-2] == 1 for s in SPLITS for (_, m) in ids[s]))
    check("C3 first utterance of each dialogue has no context", all((len(CTX["train"][i]) == 0) for i in np.flatnonzero(~pd.Series(groups_train).duplicated().values)))
    failed = [r for r in res if not r["passed"]]
    write_json(VERIF_PATH, {"n_tests": len(res), "n_failed": len(failed), "results": res})
    assert not failed, f"{len(failed)} check(s) failed: {[r['test'] for r in failed]}  -- config.json and the feature files were NOT changed"

    for s in SPLITS:
        path = os.path.join(FEATURES_DIR, FT_FILE.format(split=s)); tmp = path + ".tmp.npy"
        np.save(tmp, FT[s]); os.replace(tmp, path)
    def sha1_file(p):
        h = hashlib.sha1()
        with open(p, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()
    write_json(META_PATH, {"enabled": True, "dim": DIM, "settings": SETTINGS, "settings_id": SETTINGS_ID, "z_mu": MU.tolist(), "z_sd": SD.tolist(),
                           "frozen_sha1": {s: sha1_file(os.path.join(FEATURES_DIR, f"text_features_{s}.npy")) for s in SPLITS}})
    write_json(REPORT_PATH, rep)
    config = json.load(open(CONFIG_PATH))                                         
    config.update({"text_feature_file": FT_FILE, "text_fusion_feature_dim": DIM,
                   "text_feature_set": f"frozen {TEXT_MODEL} mean-pooled + out-of-fold fine-tuned {FT_MODEL} log-probabilities with {CTX_N}-utterance dialogue context ({K} dialogue-grouped folds, {EPOCHS} epochs, fixed)"})
    write_json(CONFIG_PATH, config)
    print(f"\nAll {len(res)} checks passed. Saved {[FT_FILE.format(split=s) for s in SPLITS]} ({DIM}-d) and updated config.json.")
    print("Phase 3b Fine-tuned text features (out-of-fold) completed successfully. Phase 6 onward uses them automatically.")